In [ ]:
# data_preprocessing_optimized.py
# -*- coding: utf-8 -*-
"""
SensorTower-like CSV preprocessing pipeline.

What it does:
1) Build "Downloads by Source (app-wise)" from wide CSVs (melt -> pivot).
2) Merge Category/Rank, DAU, ARPDAU/Revenue with consistent keys (App ID, Date).
3) Left-join everything onto Downloads base; normalize types, fill key columns.
4) Gentle imputations for DAU and revenue-like metrics; final augmented outputs.

Usage (Windows paths are fine):
  python data_preprocessing_optimized.py ^
    --root "C:/Users/you/anaconda_projects/Merging folder" ^
    --downloads "Google Play Download Source by Absolute Downloads*.csv" ^
    --category  "Google Play Category Rankings*.csv" ^
    --dau       "Google Play DAU*.csv" ^
    --arpdau    "Google Play ARPDAU*.csv"

Outputs:
  <root>/
    GooglePlay_DownloadSources_Appwise.csv
    GooglePlay_AllMerged_LEFT.csv
    GooglePlay_AllMerged_LEFT_2023.csv
    GooglePlay_AllMerged_FINAL_2021.csv
    GooglePlay_AllMerged_FINAL_2021_augmented.csv
"""

from __future__ import annotations
import argparse, glob, os, re
from pathlib import Path
from typing import List
import pandas as pd
import numpy as np

# ------------------------- utils -------------------------
def _norm_app_id(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()

def _norm_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").dt.normalize()

def _ensure_cols(df: pd.DataFrame, cols: List[str]) -> None:
    for c in cols:
        if c not in df.columns:
            df[c] = 0

def _read_csv_smart(path: str, **kw) -> pd.DataFrame:
    """Read CSV with safer defaults (thousands=',' handled in caller if needed)."""
    df = pd.read_csv(path, **kw)
    return df

def _save_csv(df: pd.DataFrame, p: Path) -> None:
    p.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(p, index=False, encoding="utf-8")

# --------------------- 1) Downloads by Source ---------------------
def build_downloads_appwise(root: Path, pattern: str) -> Path:
    files = sorted(glob.glob(str(root / pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched: {pattern}")

    all_long = []
    src_re = r"(Organic Search|Organic Browse|Paid Ads|Paid Search|Web Browser)"
    for f in files:
        dfw = _read_csv_smart(f, sep=",", thousands=",")
        # melt into long
        df_long = dfw.melt(id_vars="Date", var_name="app_col", value_name="Downloads")
        # extract source + app name
        df_long["Traffic Source"] = df_long["app_col"].str.extract(src_re)
        df_long["App Name"] = df_long["app_col"].str.replace(
            fr"\s*{src_re}\s*Downloads\s*$", "", regex=True
        )
        df_long = df_long[["Date", "App Name", "Traffic Source", "Downloads"]].copy()
        df_long["Date"] = _norm_date(df_long["Date"])
        df_long["Downloads"] = pd.to_numeric(df_long["Downloads"], errors="coerce").fillna(0).round().astype("Int64")
        all_long.append(df_long)

    merged = pd.concat(all_long, ignore_index=True)

    # pivot back to wide app-wise (per Date, App Name)
    df_pivot = merged.pivot_table(
        index=["Date", "App Name"],
        columns="Traffic Source",
        values="Downloads",
        aggfunc="sum",
        fill_value=0
    ).reset_index()
    df_pivot.columns.name = None
    _ensure_cols(df_pivot, ["Organic Search", "Organic Browse", "Paid Ads", "Paid Search", "Web Browser"])

    out = root / "GooglePlay_DownloadSources_Appwise.csv"
    _save_csv(df_pivot, out)
    return out

# --------------------- 2) Category Rankings (Rank/Category) ---------------------
def build_category_rankings(root: Path, pattern: str) -> Path:
    files = sorted(glob.glob(str(root / pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched: {pattern}")

    all_cat = []
    for f in files:
        # raw often has utf-16 + tab + one header-row to skip
        df = _read_csv_smart(f, encoding="utf-16", sep="\t", skiprows=1)
        # first data row contains category in 'topselling_free'
        # keep robust in case column name varies in capitalization
        col_candidates = [c for c in df.columns if c.lower() == "topselling_free"]
        category = df[col_candidates[0]].iloc[0] if col_candidates else None
        # drop that first row; keep the rest
        df = df.iloc[1:].copy()
        if category is not None:
            df["Category"] = category
        all_cat.append(df)

    cat = pd.concat(all_cat, ignore_index=True)
    # normalize keys if present
    if "Date" in cat.columns:
        cat["Date"] = _norm_date(cat["Date"])
    if "App ID" in cat.columns:
        cat["App ID"] = _norm_app_id(cat["App ID"])

    out = root / "GooglePlay_CategoryRankings_Merged.csv"
    _save_csv(cat, out)
    return out

# --------------------- 3) DAU / ARPDAU ---------------------
def build_metric_merged(root: Path, pattern: str, out_name: str) -> Path:
    files = sorted(glob.glob(str(root / pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched: {pattern}")

    all_df = []
    for f in files:
        df = _read_csv_smart(f, encoding="utf-16", sep="\t")
        all_df.append(df)

    df = pd.concat(all_df, ignore_index=True)
    if "Date" in df.columns:
        df["Date"] = _norm_date(df["Date"])
    if "App ID" in df.columns:
        df["App ID"] = _norm_app_id(df["App ID"])

    out = root / out_name
    _save_csv(df, out)
    return out

# --------------------- 4) Normalization helper ---------------------
def load_norm(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.rename(columns={"app id": "App ID", "app name": "App Name"}, errors="ignore")
    if "App ID" in df.columns:
        df["App ID"] = _norm_app_id(df["App ID"])
    if "Date" in df.columns:
        df["Date"] = _norm_date(df["Date"])
    df.drop(columns=["App Name"], errors="ignore", inplace=True)
    df = df.dropna(subset=["App ID", "Date"])
    df = df[df["App ID"].str.strip() != ""]
    return df

# --------------------- 5) Left-join everything ---------------------
def merge_all_left(root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    p_downloads = root / "GooglePlay_DownloadSources_Appwise.csv"
    p_category  = root / "GooglePlay_CategoryRankings_Merged.csv"
    p_dau       = root / "GooglePlay_DAU_Merged.csv"
    p_arpdau    = root / "GooglePlay_ARPDAU_Merged.csv"

    base = load_norm(p_downloads).copy()
    cat  = load_norm(p_category)
    dau  = load_norm(p_dau)
    arpd = load_norm(p_arpdau)

    # ensure traffic components are present
    for c in ["Organic Search", "Organic Browse", "Paid Ads", "Paid Search", "Web Browser"]:
        if c in base.columns:
            base[c] = pd.to_numeric(base[c], errors="coerce").fillna(0)

    # ARPDAU / Revenue / RPD
    ar_cols = [c for c in ["Downloads", "Revenue ($)", "RPD ($)", "ARPDAU ($)"] if c in arpd.columns]
    arpd = arpd[["App ID", "Date"] + ar_cols] if ar_cols else arpd[["App ID", "Date"]]
    base = base.merge(arpd, on=["App ID", "Date"], how="left")

    # DAU
    dau_cols = [c for c in ["DAU"] if c in dau.columns]
    dau = dau[["App ID", "Date"] + dau_cols] if dau_cols else dau[["App ID", "Date"]]
    base = base.merge(dau, on=["App ID", "Date"], how="left")

    # Category/Rank
    cat_cols = [c for c in ["Rank", "Updated", "Category", "WasRanked"] if c in cat.columns]
    cat = cat[["App ID", "Date"] + cat_cols] if cat_cols else cat[["App ID", "Date"]]
    base = base.merge(cat, on=["App ID", "Date"], how="left")

    base = base.drop_duplicates(subset=["App ID", "Date"])
    base_2023 = base[base["Date"] >= pd.Timestamp("2023-01-01")].copy()
    return base, base_2023

# --------------------- 6) Filling rules ---------------------
def fill_dau_group(s: pd.Series) -> pd.Series:
    """DAU: edges to 0, mid NAs -> rolling median(7), then median, then 0."""
    if s.notna().sum() == 0:
        return s.fillna(0).astype("Int64")
    notna = s.notna()
    leading_na = ~notna.cummax()
    trailing_na = (~notna[::-1].cummax())[::-1]
    edge_na = leading_na | trailing_na
    mid_na = s.isna() & ~edge_na

    s.loc[edge_na] = 0
    if mid_na.any():
        roll_med = s.rolling(window=7, min_periods=1, center=True).median()
        s.loc[mid_na] = roll_med.loc[mid_na]
    if s.isna().any():
        s = s.fillna(s.median())
    s = s.fillna(0)
    try:
        s = s.round().astype("Int64")
    except Exception:
        pass
    return s

def fill_rev_like_group(metric: pd.Series, dau: pd.Series, global_median: float) -> pd.Series:
    """
    Revenue-like columns (Revenue, RPD, ARPDAU):
    - if DAU==0 and NaN -> 0
    - else: rolling median(7, center) -> median per App ID -> global median
    """
    s = metric.copy()
    dau_vals = dau.astype(float).fillna(0.0).values if dau is not None else None

    if dau_vals is not None:
        mask_zero_dau_nan = (pd.Series(dau_vals, index=s.index) == 0) & s.isna()
        s.loc[mask_zero_dau_nan] = 0

    if s.notna().any():
        roll_med = s.rolling(window=7, min_periods=1, center=True).median()
        s = s.fillna(roll_med)
        s = s.fillna(s.median())

    if s.isna().any():
        s = s.fillna(global_median)

    return s

def finalize_and_save(root: Path, base: pd.DataFrame, base_2023: pd.DataFrame) -> None:
    # Drop cosmetic/name columns, Downloads (wide), App Name (if any)
    drop_cols = {
        "name_short","name clean","name_clean","name short","Name Short",
        "Downloads","App Name"
    }
    keep = [c for c in base.columns if c not in drop_cols]
    base = base[keep].copy()

    # Ensure keys + sort
    base["Date"] = _norm_date(base["Date"])
    base["App ID"] = _norm_app_id(base["App ID"])
    base = base.sort_values(["App ID", "Date"])

    # DAU fill
    if "DAU" in base.columns:
        base["DAU"] = base.groupby("App ID", group_keys=False)["DAU"].apply(fill_dau_group)
    else:
        base["DAU"] = pd.Series([pd.NA]*len(base), dtype="Int64")

    # Revenue-like
    rev_like_cols = [c for c in ["Revenue ($)","RPD ($)","ARPDAU ($)"] if c in base.columns]
    global_meds = {c: pd.to_numeric(base[c], errors="coerce").median() for c in rev_like_cols}
    for col in rev_like_cols:
        base[col] = base.groupby("App ID", group_keys=False).apply(
            lambda g: fill_rev_like_group(pd.to_numeric(g[col], errors="coerce"), g["DAU"], global_meds[col])
        ).reset_index(level=0, drop=True)

    # WasRanked: 1 if any Rank for App ID
    if "Rank" in base.columns:
        has_rank = base.groupby("App ID")["Rank"].apply(lambda s: int(pd.to_numeric(s, errors="coerce").notna().any()))
        base = base.merge(has_rank.rename("WasRanked_new"), on="App ID", how="left")
        base["WasRanked"] = base.get("WasRanked_new", 0).fillna(0).astype("int8")
        base.drop(columns=["WasRanked_new"], inplace=True, errors="ignore")

    # Save LEFT + 2023+ snapshots
    out_full  = root / "GooglePlay_AllMerged_LEFT.csv"
    out_2023  = root / "GooglePlay_AllMerged_LEFT_2023.csv"
    _save_csv(base, out_full)
    _save_csv(base_2023, out_2023)

    # FINAL (post-fill) for 2021 naming compatibility
    out_final = root / "GooglePlay_AllMerged_FINAL_2021.csv"
    _save_csv(base, out_final)

    # Add Organic/Paid totals
    df_aug = base.copy()
    for col in ["Organic Search", "Organic Browse", "Paid Ads", "Paid Search", "Web Browser"]:
        if col in df_aug.columns:
            df_aug[col] = pd.to_numeric(df_aug[col], errors="coerce").fillna(0)

    df_aug["Organic Traffic"] = df_aug.get("Organic Search", 0) + df_aug.get("Organic Browse", 0)
    df_aug["Paid Traffic"]    = df_aug.get("Paid Ads", 0) + df_aug.get("Paid Search", 0) + df_aug.get("Web Browser", 0)

    out_aug = root / "GooglePlay_AllMerged_FINAL_2021_augmented.csv"
    _save_csv(df_aug, out_aug)

# --------------------- main ---------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True, help="Root folder with CSVs (Windows path OK).")
    ap.add_argument("--downloads", default="Google Play Download Source by Absolute Downloads*.csv")
    ap.add_argument("--category",  default="Google Play Category Rankings*.csv")
    ap.add_argument("--dau",       default="Google Play DAU*.csv")
    ap.add_argument("--arpdau",    default="Google Play ARPDAU*.csv")
    args = ap.parse_args()

    root = Path(args.root)

    # Step A: construct the app-wise downloads file (melt -> pivot)
    build_downloads_appwise(root, args.downloads)

    # Step B: build helper merges
    build_category_rankings(root, args.category)
    build_metric_merged(root, args.dau,    "GooglePlay_DAU_Merged.csv")
    build_metric_merged(root, args.arpdau, "GooglePlay_ARPDAU_Merged.csv")

    # Step C: left-join everything and finalize
    base, base_2023 = merge_all_left(root)
    finalize_and_save(root, base, base_2023)

if __name__ == "__main__":
    main()
